# Student Academic Risk Prediction
## Notebook 02: Data Cleaning & Preprocessing
**Goal:** Validate, clean, encode, and scale data for ML readiness

### Imports & Load


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

# Load combined dataset
df = pd.read_csv('../data/processed/student_combined.csv')
print(f"Dataset loaded: {df.shape}")
df.head(3)

Dataset loaded: (1044, 35)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,at_risk,subject
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6,5,6,6,1,math
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4,5,5,6,1,math
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10,7,8,10,0,math


### Column Type Audit


In [2]:
# Separate columns by type — critical before any preprocessing
categorical_cols = df.select_dtypes(include='object').columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove target and grade columns from feature lists
exclude = ['at_risk', 'G1', 'G2', 'G3']
numerical_cols = [c for c in numerical_cols if c not in exclude]

print(f"Categorical columns ({len(categorical_cols)}):\n{categorical_cols}")
print(f"\nNumerical columns ({len(numerical_cols)}):\n{numerical_cols}")

Categorical columns (18):
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'subject']

Numerical columns (13):
['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']


#### Observation: Column Types

After auditing the columns I can see two clear groups:

- **Categorical columns** contain text labels like `yes/no`, `mother/father`, 
  `urban/rural` — these need to be converted to numbers before any ML model 
  can read them.

- **Numerical columns** contain counts and scores like study time, absences, 
  and age — these are already numeric but need scaling so no single feature 
  dominates due to its range.

I deliberately excluded G1, G2, G3 and at_risk from both lists. G3 was used 
to create my target variable. G1 and G2 are interim grades — I'll address 
those separately as a data leakage concern.

### Outlier Detection


In [3]:
# Use IQR method to detect outliers in numerical features
def detect_outliers_iqr(df, columns):
    outlier_report = []
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower) | (df[col] > upper)]
        outlier_report.append({
            'feature': col,
            'outlier_count': len(outliers),
            'lower_bound': round(lower, 2),
            'upper_bound': round(upper, 2)
        })
    return pd.DataFrame(outlier_report).sort_values('outlier_count', ascending=False)

outlier_df = detect_outliers_iqr(df, numerical_cols)
print(outlier_df.to_string(index=False))

   feature  outlier_count  lower_bound  upper_bound
  failures            183          0.0          0.0
    famrel             77          2.5          6.5
  freetime             64          1.5          5.5
 studytime             62         -0.5          3.5
  absences             54         -9.0         15.0
      Dalc             52         -0.5          3.5
traveltime             24         -0.5          3.5
       age              2         13.0         21.0
      Medu              0         -1.0          7.0
      Fedu              0         -2.0          6.0
     goout              0         -1.0          7.0
      Walc              0         -2.0          6.0
    health              0          0.0          8.0


#### Decision: Handling Outliers

After reviewing the outlier report I've decided **not to remove outliers** 
from this dataset. Here's my reasoning:

1. This is student data — a student with 90 absences or 0 study hours 
   is not a data error. That IS the signal. Removing them would remove 
   exactly the at-risk students I'm trying to detect.

2. Tree-based models like Random Forest and XGBoost are naturally 
   robust to outliers — they split on thresholds, not distances.

3. The dataset is small (1,044 rows). Removing outliers would cost 
   me valuable training examples.

I will keep all records and let the model learn from the full distribution.

### Encode Categorical Variables


In [5]:
# Label encode binary categoricals
# One-hot encode multi-class categoricals
df_clean = df.copy()

# Binary columns (yes/no type)
binary_cols = [col for col in categorical_cols 
               if df[col].nunique() == 2]

# Multi-class columns
multiclass_cols = [col for col in categorical_cols 
                   if df[col].nunique() > 2]

print(f"Binary cols to label encode: {binary_cols}")
print(f"Multi-class cols to one-hot encode: {multiclass_cols}")

# Label encode binary
le = LabelEncoder()
for col in binary_cols:
    df_clean[col] = le.fit_transform(df_clean[col])

# One-hot encode multi-class
df_clean = pd.get_dummies(df_clean, columns=multiclass_cols, drop_first=True)

print(f"\nShape after encoding: {df_clean.shape}")

Binary cols to label encode: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'subject']
Multi-class cols to one-hot encode: ['Mjob', 'Fjob', 'reason', 'guardian']

Shape after encoding: (1044, 44)


#### Decision: Encoding Strategy

I used two different encoding strategies depending on the number of 
unique values in each categorical column:

- **Label Encoding** for binary columns (e.g. sex: M/F → 1/0). 
  With only two values, a single number is sufficient and clean.

- **One-Hot Encoding** for multi-class columns (e.g. Mjob with 5 
  job categories). This creates a separate binary column for each 
  category so the model doesn't assume any ordering between them 
  (e.g. it shouldn't assume "teacher" > "health" numerically).

I used `drop_first=True` to avoid the **dummy variable trap** — 
if I have 5 job categories and keep all 5 dummies, one is always 
perfectly predictable from the others, which causes multicollinearity.

### Drop Leaky Features


In [6]:
# G1 and G2 are interim grades — highly predictive but not available
# at the START of semester. Using them would be data leakage.
# We keep them for one model variant but drop for the primary model.

leaky_features = ['G1', 'G2', 'G3']
df_clean = df_clean.drop(columns=leaky_features)

print(f"Dropped leaky features: {leaky_features}")
print(f"Final shape: {df_clean.shape}")
print(f"\nRemaining columns:\n{list(df_clean.columns)}")

Dropped leaky features: ['G1', 'G2', 'G3']
Final shape: (1044, 41)

Remaining columns:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'at_risk', 'subject', 'Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher', 'Fjob_health', 'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_home', 'reason_other', 'reason_reputation', 'guardian_mother', 'guardian_other']


#### Decision: Removing Data Leakage

This is one of the most important decisions in this project.

G1 and G2 are first and second period grades. They are extremely 
predictive of G3 — but they are collected *during* the school year. 
An early warning system must make predictions *before* those grades 
exist, otherwise it's useless in practice.

Keeping G1 and G2 would give me artificially high model accuracy 
but a system that can't actually be deployed. This is called 
**data leakage** — when information from the future leaks into 
training data.

I drop all three grade columns. My model will rely entirely on 
demographic, socioeconomic, and behavioral features.

### Scale Numerical Features


In [10]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Only scale numerical features, not encoded binaries
cols_to_scale = [c for c in numerical_cols if c in df_clean.columns]

df_clean[cols_to_scale] = scaler.fit_transform(df_clean[cols_to_scale])

print(f"Scaled {len(cols_to_scale)} numerical features:")
print(cols_to_scale)
print(f"\nSample after scaling:")
df_clean[cols_to_scale].describe().round(2)

Scaled 13 numerical features:
['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']

Sample after scaling:


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences
count,1044.00,1044.00,1044.00,1044.00,1044.00,1044.00,1044.00,1044.00,1044.00,1044.00,1044.00,1044.00,1044.00
mean,0.00,-0.00,-0.00,-0.00,0.00,0.00,-0.00,0.00,0.00,0.00,0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.39,-2.32,-2.17,-0.72,-1.16,-0.40,-3.15,-2.13,-1.87,-0.54,-1.00,-1.79,-0.71
25%,-0.59,-0.54,-1.26,-0.72,-1.16,-0.40,0.07,-0.20,-1.00,-0.54,-1.00,-0.38,-0.71
50%,0.22,0.35,-0.35,-0.72,0.04,-0.40,0.07,-0.20,-0.14,-0.54,-0.22,0.32,-0.39
75%,1.03,1.24,0.56,0.65,0.04,-0.40,1.14,0.77,0.73,0.55,0.56,1.02,0.25
max,4.26,1.24,1.47,3.39,2.43,4.17,1.14,1.74,1.60,3.85,2.11,1.02,11.37


In [12]:
# Save Cleaned Data

df_clean.to_csv('../data/processed/student_clean.csv', index=False)

print(f"Cleaned dataset saved.")
print(f"Final shape: {df_clean.shape}")
print(f"Target distribution:\n{df_clean['at_risk'].value_counts()}")

Cleaned dataset saved.
Final shape: (1044, 41)
Target distribution:
at_risk
0    814
1    230
Name: count, dtype: int64
